In [ ]:
from __future__ import print_function

from pyspark.ml.regression import DecisionTreeRegressor, LinearRegression
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import numpy as np
import pandas as pd

In [ ]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("GWU-project1") \
    .master("local[1]") \
    .getOrCreate()


In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true")\
    .csv("data/Life_Expectancy_Data3.csv")

In [ ]:
df.show(1)

In [ ]:
df.printSchema()

In [ ]:
#####################################################################################################################
#####################################################################################################################

In [ ]:
df.createOrReplaceTempView("life_expectancy_data")

# SQL query to remove rows with any NaN or NULL values in the DataFrame
clean_query = """
SELECT *
FROM life_expectancy_data
WHERE NOT (
    {conditions}
)
"""

# Generate conditions for each column to check for NULL or NaN (for numerical columns)
columns = df.columns
conditions = " OR ".join([f"`{col}` IS NULL OR isnan(`{col}`)" if "double" in str(df.schema[col].dataType) else f"`{col}` IS NULL" for col in columns])

# Complete and run the query
clean_query = clean_query.format(conditions=conditions)
df_clean = spark.sql(clean_query)

# Register the clean DataFrame for further operations
df_clean.createOrReplaceTempView("clean_life_expectancy_data")

# Check and show how many rows remain
df_clean.show()


In [ ]:
# Register the DataFrame as a temporary view if not already done
df_clean.createOrReplaceTempView("clean_life_expectancy_data")

# Generate conditions for checking NaN or NULL in each column
columns = df_clean.columns
conditions = " OR ".join([f"`{col}` IS NULL OR isnan(`{col}`)" if "double" in str(df.schema[col].dataType) else f"`{col}` IS NULL" for col in columns])

# SQL query to count rows that have any NaN or NULL values
nan_null_count_query = f"""
SELECT COUNT(*) AS count_nan_null_rows
FROM clean_life_expectancy_data
WHERE {conditions}
"""

# Execute the query
nan_null_counts = spark.sql(nan_null_count_query)

# Show the result
nan_null_counts.show()


In [ ]:
# Register the cleaned DataFrame as a temporary view
df_clean.createOrReplaceTempView("clean_life_expectancy_data")

# Check the minimum and maximum of Life expectancy to decide bin ranges
spark.sql("SELECT MIN(`Life`), MAX(`Life`) FROM clean_life_expectancy_data").show()



In [ ]:
# SQL query to create histogram data for the "Life" feature with an additional category for outliers or NULLs
histogram_query = """
SELECT 
    CASE
        WHEN FLOOR(`Life`) BETWEEN 44 AND 49 THEN '44-49'
        WHEN FLOOR(`Life`) BETWEEN 50 AND 54 THEN '50-54'
        WHEN FLOOR(`Life`) BETWEEN 55 AND 59 THEN '55-59'
        WHEN FLOOR(`Life`) BETWEEN 60 AND 64 THEN '60-64'
        WHEN FLOOR(`Life`) BETWEEN 65 AND 69 THEN '65-69'
        WHEN FLOOR(`Life`) BETWEEN 70 AND 74 THEN '70-74'
        WHEN FLOOR(`Life`) BETWEEN 75 AND 79 THEN '75-79'
        WHEN FLOOR(`Life`) BETWEEN 80 AND 84 THEN '80-84'
        WHEN FLOOR(`Life`) BETWEEN 85 AND 89 THEN '85-89'
        ELSE 'Out of Range or NULL' END AS `Life_Expectancy_Range`,
    COUNT(*) AS `Count`
FROM 
    clean_life_expectancy_data
GROUP BY 
    `Life_Expectancy_Range`
ORDER BY 
    `Life_Expectancy_Range`
"""

# Execute the SQL query
histogram_data = spark.sql(histogram_query)

# Show the results
histogram_data.show(truncate=False)


In [ ]:
import matplotlib.pyplot as plt

# Convert the Spark DataFrame to a Pandas DataFrame
histogram_data_pd = histogram_data.toPandas()

# Plotting the histogram
plt.figure(figsize=(10, 6))
plt.bar(histogram_data_pd['Life_Expectancy_Range'], histogram_data_pd['Count'], color='skyblue')
plt.xlabel('Life Expectancy Range')
plt.ylabel('Count')
plt.title('Histogram of Life Expectancy Distribution')
plt.xticks(rotation=45)  # Rotate the x-axis labels for better readability
plt.grid(False)  # Add a grid for easier visualization
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Convert Spark DataFrame to Pandas DataFrame
df_pandas = df_clean.toPandas()

# Group by "Country" and "Region", and calculate the mean of "GDP", "Life expectancy", and "Population"
df_country = df_pandas.groupby(["Country", "Region"])[["GDP", "Life", "Population"]].mean()

# Drop rows with NaN values and reset index
df_country = df_country.dropna().reset_index()

# Set the plot size
plt.figure(figsize=(12, 8))

# Create a scatter plot
g = sns.relplot(x="GDP", y="Life", size="Population", sizes=(50, 600),
                data=df_country, height=6, aspect=1.2, alpha=0.8, hue="Region")

# Setting the x-axis to logarithmic scale
plt.xscale("log")

# Adding title and labels
plt.title('GDP and Life Expectancy by Country', fontsize=16)
plt.xlabel('Log of GDP per Capita')
plt.ylabel('Life Expectancy')

# Display the plot
plt.show()


In [ ]:
from pyspark.ml.feature import StringIndexer

# Assuming 'Region' is the column to be indexed
indexer = StringIndexer(inputCol="Region", outputCol="RegionCode")
df_clean = indexer.fit(df_clean).transform(df_clean)

from pyspark.ml.feature import VectorAssembler

# Drop unnecessary columns
columns_to_drop = ["Country", "Status", "Region"]
df_clean = df_clean.drop(*columns_to_drop)

# Assuming all remaining columns are relevant for PCA and are numerical
features = [col for col in df_clean.columns if col != 'RegionCode']

# Create a features vector
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_clean = assembler.transform(df_clean)


In [ ]:
from pyspark.ml.feature import StandardScaler

# Standardize the features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)
scalerModel = scaler.fit(df_clean)
df_clean = scalerModel.transform(df_clean)

from pyspark.ml.feature import PCA

# Apply PCA
pca = PCA(k=2, inputCol="scaledFeatures", outputCol="pcaFeatures")
model = pca.fit(df_clean)
result = model.transform(df_clean)

# Project the data to principal components
pca_result = result.select("pcaFeatures")
pca_result.show(truncate=False)


In [ ]:
# Assuming 'RegionCode' is still in the Spark DataFrame 'result'
full_pca_result_pd = result.select("pcaFeatures", "RegionCode").toPandas()
full_pca_result_pd['PCA1'] = full_pca_result_pd['pcaFeatures'].apply(lambda x: x[0])
full_pca_result_pd['PCA2'] = full_pca_result_pd['pcaFeatures'].apply(lambda x: x[1])

# Now plot with color coding
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.scatterplot(x='PCA1', y='PCA2', hue='RegionCode', palette='viridis', data=full_pca_result_pd, alpha=0.7)
plt.title('PCA by Region')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()


In [ ]:
##################################################################################################################
##################################################################################################################

In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true")\
    .csv("data/Life_Expectancy_Data3.csv")

In [ ]:
df.printSchema()

In [ ]:
# Cleaning the Dataset REMOVE Two Columns
df = df.drop(*['Year','Country'])             # dropping some irrelevant columns


In [ ]:
# IMPUTER
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=df.drop('Status','Region').columns,
                 outputCols=["{}".format(c) for c in df.drop('Status','Region').columns]).setStrategy('mean')

# add imputation to cols
df_imputed = imputer.fit(df).transform(df)

In [ ]:
df_imputed.columns

In [ ]:
# Find Count of Null, None, NaN of All DataFrame Columns
from pyspark.sql.functions import col,isnan, when, count
df_imputed.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df_imputed.columns]
   ).show()

In [ ]:
#One hot enconde Status and Region

# First we index it
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCols=['Region','Status'], outputCols=['Region_Numeric','Status_Numeric'])
indexer_fitted = indexer.fit(df_imputed)
df_indexed = indexer_fitted.transform(df_imputed)

#now we can one hot encode
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder(inputCols=['Region_Numeric','Status_Numeric'], outputCols=['Region_onehot','Status_onehot'])
df_onehot = encoder.fit(df_indexed).transform(df_indexed)

#Remobe the categorical columns not needed anymore
df_onehot = df_onehot.drop(*['Status','Region','Region_Numeric','Status_Nhmeric'])
df_onehot.columns

In [ ]:
df_onehot.printSchema()

In [ ]:
input_cols = df_onehot.drop('Life').columns

In [ ]:
assembler = VectorAssembler(inputCols=input_cols, outputCol='features')

df_transform = assembler.transform(df_onehot).select("Life", "features")

In [ ]:
df_transform.show(5)

In [ ]:
from pyspark.ml.feature import Normalizer, StandardScaler

scaler = StandardScaler(inputCol='features', outputCol='scaled_features')

df_scaled = scaler.fit(df_transform).transform(df_transform)

In [ ]:
df_scaled.show(5)

In [ ]:
# Let's split our data into training data and testing data
train,test = df_scaled.randomSplit([0.8,0.2])

In [ ]:
##############################################################

lr_model = LinearRegression(featuresCol='scaled_features', labelCol='Life')
lr_fitted = lr_model.fit(train)

In [ ]:
trainingSummary = lr_fitted.summary
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

In [ ]:
# Now create our decision tree
dtr = DecisionTreeRegressor().setFeaturesCol("scaled_features").setLabelCol("Life")

# Train the model using our training data
tree_model = dtr.fit(train)

In [ ]:
# Now see if we can predict values in our test data.
# Generate predictions using our decision tree model for all features in our
# test dataframe:
fullPredictions = tree_model.transform(test).cache()

In [ ]:
fullPredictions.show()

In [ ]:
dt_evaluator = RegressionEvaluator(labelCol="Life", predictionCol="prediction", metricName="r2")

rmse = dt_evaluator.evaluate(fullPredictions)

print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

In [ ]:
lr=LinearRegression(featuresCol='scaled_features', labelCol='Life')

In [ ]:
pipeline1=Pipeline(stages=[lr])

In [ ]:
paramgrid =ParamGridBuilder()\
.addGrid(lr.regParam, [0, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0])\
.addGrid(lr.maxIter, [10])\
.build()


evaluator=RegressionEvaluator(metricName="r2").setLabelCol("Life")


crossval= CrossValidator(estimator=pipeline1,  
                         estimatorParamMaps=paramgrid,
                         evaluator = evaluator, 
                         numFolds=5
                        )

cvModel1=crossval.fit(train) 
evaluator.evaluate(cvModel1.transform(test))


In [ ]:
cvModel1.getEstimatorParamMaps()[ np.argmax(cvModel1.avgMetrics) ]

In [ ]:
cvModel1.avgMetrics

In [ ]:
rf = RandomForestRegressor(featuresCol='scaled_features', labelCol='Life')
pipeline2 = Pipeline(stages=[rf])

paramgrid =ParamGridBuilder()\
.addGrid(rf.numTrees, [200, 500])\
.addGrid(rf.maxDepth, [5,10])\
.build()


evaluator=RegressionEvaluator(metricName="r2").setLabelCol("Life")


crossval= CrossValidator(estimator=pipeline2,  
                         estimatorParamMaps=paramgrid,
                         evaluator = evaluator, 
                         numFolds=5
                        )

cvModel2=crossval.fit(train) 

evaluator.evaluate(cvModel2.transform(test))


In [ ]:
cvModel2.getEstimatorParamMaps()[ np.argmax(cvModel2.avgMetrics) ]

In [ ]:
gbt = GBTRegressor(featuresCol='scaled_features', labelCol='Life')
pipeline3 = Pipeline(stages=[gbt])

paramGrid = ParamGridBuilder()\
  .addGrid(gbt.maxDepth, [2, 5, 10])\
  .addGrid(gbt.maxIter, [10,100,500])\
  .build()

evaluator=RegressionEvaluator(metricName="r2").setLabelCol("Life")


crossval= CrossValidator(estimator=pipeline3,  
                         estimatorParamMaps=paramgrid,
                         evaluator = evaluator, 
                         numFolds=5
                        )

cvModel3=crossval.fit(train) 

evaluator.evaluate(cvModel3.transform(test)